Self-attention implementation

In [46]:
# Dummy self-attention
import torch

inputs = torch.tensor(
[
    [0.43, 0.15, 0.89],     # Your (x^1)
    [0.55, 0.87, 0.66],     # journey (x^2)
    [0.57, 0.85, 0.64],     # starts (x^3)
    [0.22, 0.58, 0.33],     # with (x^4)
    [0.77, 0.25, 0.10],     # one (x^5)
    [0.05, 0.80, 0.55]
] # step (x^6)
)

In [47]:
query = inputs[1]

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
attn_scores_2

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

In [48]:
# Scalar product

res = 0
for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]
    

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


In [49]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [50]:
torch.dot(inputs[2], inputs[2])

tensor(1.4570)

In [51]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)


attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [52]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [53]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for idx, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
context_vec_2

tensor([0.4095, 0.5534, 0.5012])

In [54]:
attn_scores = torch.empty(inputs.shape[0], inputs.shape[0])
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [55]:
attn_scores = inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [56]:
attn_weights = torch.softmax(attn_scores, dim = -1)
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [57]:
attn_weights.sum(dim=-1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [58]:
all_context_vecs = attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [59]:
### Initial transformer - scaled dot-product attention

x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [60]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [61]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
query_2

tensor([0.4306, 1.4551])

In [62]:
keys = inputs @ W_key
values = inputs @ W_value
keys.shape, values.shape

(torch.Size([6, 2]), torch.Size([6, 2]))

In [63]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
attn_score_22

tensor(1.6307)

In [64]:
attn_scores_2 - query_2 @ keys.T
attn_scores_2

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

In [65]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
attn_weights_2

tensor([0.1476, 0.2164, 0.2134, 0.1365, 0.1240, 0.1621])

In [66]:
context_vec_2 = attn_weights_2 @ values
context_vec_2

tensor([0.3518, 0.8867])

In [67]:
### Single class
import torch.nn as nn


class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
    
    def forward(self, x):
        keys = x @ self.W_key
        values = x @ self.W_value
        queries = x @ self.W_query
        attn_scores = queries @ keys.T  # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [68]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
sa_v1(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [69]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [70]:
torch.manual_seed(451)

sa_v2 = SelfAttention_v2(d_in, d_out)
sa_v2(inputs)

tensor([[-0.1418,  0.1527],
        [-0.1411,  0.1569],
        [-0.1411,  0.1568],
        [-0.1409,  0.1572],
        [-0.1415,  0.1547],
        [-0.1407,  0.1581]], grad_fn=<MmBackward0>)

In [71]:
sa_v1.W_query.data = sa_v2.W_query.weight.T
sa_v1.W_key.data = sa_v2.W_key.weight.T
sa_v1.W_value.data = sa_v2.W_value.weight.T

print(sa_v1(inputs), sa_v2(inputs), sep='\n')

tensor([[-0.1418,  0.1527],
        [-0.1411,  0.1569],
        [-0.1411,  0.1568],
        [-0.1409,  0.1572],
        [-0.1415,  0.1547],
        [-0.1407,  0.1581]], grad_fn=<MmBackward0>)
tensor([[-0.1418,  0.1527],
        [-0.1411,  0.1569],
        [-0.1411,  0.1568],
        [-0.1409,  0.1572],
        [-0.1415,  0.1547],
        [-0.1407,  0.1581]], grad_fn=<MmBackward0>)


# Causal attention

In [72]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
attn_weights

tensor([[0.1722, 0.1670, 0.1672, 0.1628, 0.1700, 0.1609],
        [0.1624, 0.1631, 0.1632, 0.1718, 0.1692, 0.1704],
        [0.1626, 0.1631, 0.1632, 0.1716, 0.1692, 0.1702],
        [0.1625, 0.1642, 0.1642, 0.1710, 0.1676, 0.1706],
        [0.1675, 0.1652, 0.1654, 0.1670, 0.1695, 0.1653],
        [0.1604, 0.1635, 0.1634, 0.1729, 0.1671, 0.1727]],
       grad_fn=<SoftmaxBackward0>)

In [73]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
mask_simple

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [74]:
masked_simple = attn_weights*mask_simple
masked_simple

tensor([[0.1722, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1624, 0.1631, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1626, 0.1631, 0.1632, 0.0000, 0.0000, 0.0000],
        [0.1625, 0.1642, 0.1642, 0.1710, 0.0000, 0.0000],
        [0.1675, 0.1652, 0.1654, 0.1670, 0.1695, 0.0000],
        [0.1604, 0.1635, 0.1634, 0.1729, 0.1671, 0.1727]],
       grad_fn=<MulBackward0>)

In [75]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
masked_simple_norm

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4989, 0.5011, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3325, 0.3337, 0.3338, 0.0000, 0.0000, 0.0000],
        [0.2455, 0.2480, 0.2480, 0.2584, 0.0000, 0.0000],
        [0.2007, 0.1980, 0.1982, 0.2001, 0.2031, 0.0000],
        [0.1604, 0.1635, 0.1634, 0.1729, 0.1671, 0.1727]],
       grad_fn=<DivBackward0>)

In [76]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[ 0.0871,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.1470, -0.1408,    -inf,    -inf,    -inf,    -inf],
        [-0.1426, -0.1375, -0.1368,    -inf,    -inf,    -inf],
        [-0.1192, -0.1046, -0.1046, -0.0466,    -inf,    -inf],
        [-0.0212, -0.0407, -0.0394, -0.0259, -0.0046,    -inf],
        [-0.1660, -0.1394, -0.1398, -0.0600, -0.1082, -0.0618]],
       grad_fn=<MaskedFillBackward0>)

In [77]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4989, 0.5011, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3325, 0.3337, 0.3338, 0.0000, 0.0000, 0.0000],
        [0.2455, 0.2480, 0.2480, 0.2584, 0.0000, 0.0000],
        [0.2007, 0.1980, 0.1982, 0.2001, 0.2031, 0.0000],
        [0.1604, 0.1635, 0.1634, 0.1729, 0.1671, 0.1727]],
       grad_fn=<SoftmaxBackward0>)

Dropout

In [78]:
torch.manual_seed('123')
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6,6)
dropout(example)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [79]:
dropout(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6673, 0.6677, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.4961, 0.5168, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.4062, 0.0000],
        [0.3208, 0.3269, 0.0000, 0.3458, 0.3342, 0.3454]],
       grad_fn=<MulBackward0>)

In [80]:
batch = torch.stack((inputs, inputs), dim=0)
batch.shape

torch.Size([2, 6, 3])

In [81]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length),
            diagonal=1)
        )
        
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        values = self.W_value(x)
        queires = self.W_query(x)
        
        attn_scores = queires @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec

In [82]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0)
context_vecs = ca(batch)
context_vecs.shape

torch.Size([2, 6, 2])

# Multi-head attention

In [83]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                 dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(
                d_in, d_out, context_length, dropout, qkv_bias
            )
             for _ in range(num_heads)]
        )
    
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=1)

In [84]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out,
                 context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens] 
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )
        context_vec = self.out_proj(context_vec)
        return context_vec


In [85]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
context_vecs

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)

In [92]:
torch.manual_seed(123)

mha = MultiHeadAttention(
    d_in=768,
    d_out=768,
    context_length=1024,
    dropout=0.0,
    num_heads=12,
)

batch = torch.rand(2, 1024, 768)
context_vecs = mha(batch)
context_vecs

tensor([[[ 0.2410,  0.0208,  0.0829,  ..., -0.0881,  0.2225, -0.0161],
         [ 0.2158, -0.0159,  0.0330,  ..., -0.0018,  0.1817, -0.0297],
         [ 0.2658, -0.0152, -0.0238,  ...,  0.0455,  0.1202, -0.0170],
         ...,
         [ 0.2917,  0.0066, -0.0047,  ...,  0.1008,  0.1277, -0.1009],
         [ 0.2920,  0.0067, -0.0049,  ...,  0.1009,  0.1281, -0.1008],
         [ 0.2921,  0.0066, -0.0049,  ...,  0.1006,  0.1279, -0.1007]],

        [[ 0.4378,  0.0251,  0.0110,  ...,  0.0405,  0.1144, -0.0659],
         [ 0.3856,  0.0042, -0.0063,  ...,  0.0636,  0.1702, -0.1100],
         [ 0.2812,  0.0238, -0.0104,  ...,  0.0248,  0.1575, -0.0406],
         ...,
         [ 0.2974,  0.0089, -0.0042,  ...,  0.0998,  0.1302, -0.0998],
         [ 0.2973,  0.0093, -0.0040,  ...,  0.0995,  0.1303, -0.0997],
         [ 0.2976,  0.0093, -0.0040,  ...,  0.0994,  0.1303, -0.0999]]],
       grad_fn=<ViewBackward0>)